In [19]:
from pyspark.sql import SparkSession  

spark = SparkSession.builder.appName("FlightData").getOrCreate()
flight_df = (spark.read.format("csv")
             .option("header", "true")
             .option("dateFormat", "M/d/yyyy")
             .load("C:\\Users\\abhis\\OneDrive\\Documents\\Pyspark Project\\flight-time.csv")
)
flight_df.show(5)

+--------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+
| FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|CRS_DEP_TIME|DEP_TIME|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|CANCELLED|DISTANCE|
+--------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+
|1/1/2000|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1115|    1113|     1343|      5|        1400|    1348|        0|     946|
|1/1/2000|        DL|             1479|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1315|    1311|     1536|      7|        1559|    1543|        0|     946|
|1/1/2000|        DL|             1857|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1415|    1414|     1642|      9|        1721|    1651|        0|     946|
|1/1/2000|

In [20]:
flight_df.printSchema()

root
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- CRS_DEP_TIME: string (nullable = true)
 |-- DEP_TIME: string (nullable = true)
 |-- WHEELS_ON: string (nullable = true)
 |-- TAXI_IN: string (nullable = true)
 |-- CRS_ARR_TIME: string (nullable = true)
 |-- ARR_TIME: string (nullable = true)
 |-- CANCELLED: string (nullable = true)
 |-- DISTANCE: string (nullable = true)



In [21]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType , DateType, TimestampType

flight_schema = StructType([
    StructField("FL_DATE", DateType()),
    StructField("OP_CARRIER", StringType()),
    StructField("OP_CARRIER_FL_NUM", StringType()),
    StructField("ORIGIN", StringType()),
    StructField("ORIGIN_CITY_NAME", StringType()),
    StructField("DEST", StringType()),
    StructField("DEST_CITY_NAME", StringType()),
    StructField("CRS_DEP_TIME", LongType()),
    StructField("DEP_TIME", LongType()),
    StructField("WHEELS_ON", IntegerType()),
    StructField("TAXI_IN", IntegerType()),
    StructField("CRS_ARR_TIME", LongType()),
    StructField("ARR_TIME", LongType()),
    StructField("CANCELLED", IntegerType()),
    StructField("DISTANCE", IntegerType())
])

In [22]:
flight_df_with_schema_def = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(flight_schema)
    .load("C:\\Users\\abhis\\OneDrive\\Documents\\Pyspark Project\\flight-time.csv")
)
flight_df_with_schema_def.show(5)


+-------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+
|FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|CRS_DEP_TIME|DEP_TIME|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|CANCELLED|DISTANCE|
+-------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+
|   NULL|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1115|    1113|     1343|      5|        1400|    1348|        0|     946|
|   NULL|        DL|             1479|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1315|    1311|     1536|      7|        1559|    1543|        0|     946|
|   NULL|        DL|             1857|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1415|    1414|     1642|      9|        1721|    1651|        0|     946|
|   NULL|       

In [23]:
flight_df_with_schema_def.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- CRS_DEP_TIME: long (nullable = true)
 |-- DEP_TIME: long (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- CRS_ARR_TIME: long (nullable = true)
 |-- ARR_TIME: long (nullable = true)
 |-- CANCELLED: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)



Next step is to load it to a SQL table

we are using that table to transform and correcting the data types

1. Read raw data from flight_time_raw table
2. Apply transformations to time values as hour to minute interval

    1. CRS_DEP_TIME
    2. DEP_TIME
    3. WHEELS_ON
    4. CRS_ARR_TIME
    5. ARR_TIME
3. Apply transformation to TAXI_IN to make it a minute interval

In [24]:
from pyspark.sql.functions import expr

step1_df = (flight_df.withColumns({
    "CRS_DEP_TIME_HH":  expr("left(lpad(CRS_DEP_TIME, 4, '0'), 2)"),  
    "CRS_DEP_TIME_MM":  expr("right(lpad(CRS_DEP_TIME, 4, '0'), 2)"),
})
)

In [25]:
step1_df.show(5)

+--------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+---------------+---------------+
| FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|CRS_DEP_TIME|DEP_TIME|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|CANCELLED|DISTANCE|CRS_DEP_TIME_HH|CRS_DEP_TIME_MM|
+--------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+---------------+---------------+
|1/1/2000|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1115|    1113|     1343|      5|        1400|    1348|        0|     946|             11|             15|
|1/1/2000|        DL|             1479|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1315|    1311|     1536|      7|        1559|    1543|        0|     946|             13|             15|
|1/1/2000|     

In [26]:
step2_df = (
    step1_df.withColumns({
       "CRS_DEP_TIME_NEW": expr("CAST(concat(CRS_DEP_TIME_HH, ':', CRS_DEP_TIME_MM) AS INTERVAL HOUR TO MINUTE)"), 
})
)

In [27]:
step2_df.limit(5).show()

+--------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+---------------+---------------+--------------------+
| FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|CRS_DEP_TIME|DEP_TIME|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|CANCELLED|DISTANCE|CRS_DEP_TIME_HH|CRS_DEP_TIME_MM|    CRS_DEP_TIME_NEW|
+--------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+---------------+---------------+--------------------+
|1/1/2000|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1115|    1113|     1343|      5|        1400|    1348|        0|     946|             11|             15|INTERVAL '11:15' ...|
|1/1/2000|        DL|             1479|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1315|    1311|     1536|      7|     

But now we have to continue the same for the other fields. Doing one by one is not very efficient. For that we are developing a resuable function

In [31]:
def get_interval(hhmm_value):
     from pyspark.sql.functions import expr

     return expr(f"""
                 cast(concat(left(lpad({hhmm_value}, 4, '0'), 2), ':', 
                             right(lpad({hhmm_value}, 4, '0'), 2)) 
                             AS INTERVAL HOUR TO MINUTE)
                 """)

In [32]:
result_df = (
    flight_df.withColumns({
        "CRS_DEP_TIME": get_interval("CRS_DEP_TIME"),
        "DEP_TIME": get_interval("DEP_TIME"),
        "WHEELS_ON": get_interval("WHEELS_ON"),
        "CRS_ARR_TIME": get_interval("CRS_ARR_TIME"),
        "ARR_TIME": get_interval("ARR_TIME"),
        "TAXI_IN": expr("cast(TAXI_IN AS INTERVAL MINUTE)")
    })
)

In [33]:
result_df.show(5)

+--------+----------+-----------------+------+----------------+----+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------+--------+
| FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|        CRS_DEP_TIME|            DEP_TIME|           WHEELS_ON|             TAXI_IN|        CRS_ARR_TIME|            ARR_TIME|CANCELLED|DISTANCE|
+--------+----------+-----------------+------+----------------+----+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------+--------+
|1/1/2000|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|INTERVAL '11:15' ...|INTERVAL '11:13' ...|INTERVAL '13:43' ...|INTERVAL '05' MINUTE|INTERVAL '14:00' ...|INTERVAL '13:48' ...|        0|     946|
|1/1/2000|        DL|             1479|   BOS|      Boston, MA| ATL|   Atlanta, GA|I

In [34]:
result_df.printSchema()

root
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- CRS_DEP_TIME: interval hour to minute (nullable = true)
 |-- DEP_TIME: interval hour to minute (nullable = true)
 |-- WHEELS_ON: interval hour to minute (nullable = true)
 |-- TAXI_IN: interval minute (nullable = true)
 |-- CRS_ARR_TIME: interval hour to minute (nullable = true)
 |-- ARR_TIME: interval hour to minute (nullable = true)
 |-- CANCELLED: string (nullable = true)
 |-- DISTANCE: string (nullable = true)

